# Aula 12 — Baselines fortes: Naive Bayes, Logistic Regression e LinearSVC

**Antes de escolher um Transformer, compare alternativas simples e competitivas**

Na Aula 11, vimos que um Transformer pode ser adaptado por fine-tuning. Mas isso não significa que ele deva ser sempre a primeira escolha.

Nesta aula vamos comparar três modelos clássicos muito usados em classificação de texto:

- `MultinomialNB`;
- `LogisticRegression`;
- `LinearSVC`.

Todos receberão a mesma representação TF-IDF para que a comparação seja justa.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar por que baselines fortes continuam importantes;
- comparar Naive Bayes, Logistic Regression e LinearSVC;
- usar validação cruzada com `f1_macro`;
- observar custo de treino e desempenho no mesmo experimento;
- interpretar diferenças entre modelos lineares;
- reconhecer quando um modelo clássico pode ser preferível a um Transformer.


## 📘 Glossário da aula

Antes de começar, percorra as palavras-chave abaixo. Elas formam o vocabulário mínimo para acompanhar a aula:

**Palavras-chave:** [Baseline](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#baseline) · [Regressão Logística](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#regressão-logística) · [LinearSVC](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#linearsvc) · [Margem](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#margem) · [Coeficiente](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#coeficiente) · [Validação cruzada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#validação-cruzada) · [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#f1-score)

Use esses links como pontos de entrada: consulte um termo quando ele aparecer no notebook e volte à aula em seguida.

- [Abrir o Glossário Vivo completo — PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Open the full Living Glossary — EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> O notebook explica o necessário para seguir a aula; o glossário consolida o vocabulário técnico e ajuda a conectar conceitos entre aulas.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. Por que voltar aos modelos clássicos depois de BERT?

Porque um modelo mais complexo só é justificável se entregar valor adicional relevante.

Em produção, a pergunta não é apenas:

> Qual modelo tem maior capacidade?

Mas também:

```text
qualidade
↔ custo
↔ latência
↔ interpretabilidade
↔ manutenção
```

Um baseline forte ajuda a medir se a complexidade extra realmente vale a pena.


## 3. Dataset didático

Vamos reutilizar um conjunto balanceado com três classes: `duvida`, `reclamacao` e `elogio`.


In [ ]:
texts = [
    # duvida
    "como altero minha senha",
    "onde vejo minha fatura",
    "posso pagar amanhã",
    "como atualizo meu cadastro",
    "qual o prazo para resposta",
    "como cancelo o serviço",
    "onde consulto meu limite",
    "como desbloqueio minha conta",
    "posso mudar a data de vencimento",
    "como acesso a segunda via da fatura",
    "onde encontro meu extrato",
    "como faço para recuperar minha senha",

    # reclamacao
    "meu pedido não chegou",
    "o atendimento foi péssimo",
    "estou insatisfeito com o serviço",
    "a entrega atrasou novamente",
    "o suporte não resolveu meu problema",
    "estou muito irritado com o atendimento",
    "ninguém resolveu minha solicitação",
    "o serviço está muito ruim",
    "estou cansado desse atraso",
    "me cobraram um valor incorreto",
    "o aplicativo não funciona",
    "meu problema continua sem solução",

    # elogio
    "o atendimento foi excelente",
    "fui muito bem atendido",
    "serviço rápido e eficiente",
    "estou satisfeito com o atendimento",
    "a equipe resolveu tudo rapidamente",
    "gostei muito do suporte",
    "o atendimento foi ótimo",
    "excelente trabalho da equipe",
    "meu problema foi resolvido rapidamente",
    "o suporte foi muito prestativo",
    "estou muito satisfeito com o serviço",
    "parabéns pelo excelente atendimento",
]

labels = (
    ["duvida"] * 12
    + ["reclamacao"] * 12
    + ["elogio"] * 12
)

print("Documentos:", len(texts))


## 4. Mesma representação, modelos diferentes

Para comparar os classificadores de forma mais justa, todos usarão o mesmo `TfidfVectorizer`.

A única parte que muda será o classificador.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

SEED = 42

models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=SEED,
    ),
    "LinearSVC": LinearSVC(
        random_state=SEED,
    ),
}

pipelines = {
    name: Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", classifier),
    ])
    for name, classifier in models.items()
}


## 5. Validação cruzada com a mesma métrica

Vamos usar `StratifiedKFold` para preservar aproximadamente a proporção das classes em cada fold e medir `f1_macro`.


> 📘 **Glossário em contexto:** se `validação cruzada` ou `F1-score` ainda não estiverem naturais para você, consulte os links do Glossário da aula antes de executar a próxima célula.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=SEED,
)

rows = []

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        texts,
        labels,
        cv=cv,
        scoring="f1_macro",
        return_train_score=False,
    )

    rows.append({
        "model": name,
        "f1_macro_mean": scores["test_score"].mean(),
        "f1_macro_std": scores["test_score"].std(),
        "fit_time_mean_s": scores["fit_time"].mean(),
        "score_time_mean_s": scores["score_time"].mean(),
    })

results = pd.DataFrame(rows).sort_values(
    "f1_macro_mean",
    ascending=False,
)

results.round(4)


### O que observar

Não procure apenas 'quem ganhou'. Observe também:

- diferença média de F1;
- variabilidade entre folds;
- tempo de treino;
- simplicidade do pipeline.

> Em datasets pequenos, diferenças mínimas podem ser apenas ruído experimental.


### Como ler esta tabela sem cair na armadilha do "vencedor"

No nosso resultado, `LinearSVC` ficou na frente em F1 macro, mas a diferença média para os demais foi pequena:

```text
LinearSVC           ≈ 0.673
MultinomialNB       ≈ 0.657
LogisticRegression  ≈ 0.657
```

A diferença entre `LinearSVC` e os demais é de cerca de `0.016` em F1 macro.

Ao mesmo tempo, os desvios-padrão entre folds ficaram bem maiores do que essa diferença:

```text
LinearSVC           std ≈ 0.088
MultinomialNB       std ≈ 0.113
LogisticRegression  std ≈ 0.113
```

Isso sugere cautela:

> **uma média ligeiramente maior não é evidência suficiente de superioridade robusta quando a variabilidade entre folds é alta.**

Em outras palavras:

```text
ganho médio pequeno
+ variabilidade relativamente alta
→ evidência fraca de superioridade
```

O empate entre Naive Bayes e Logistic Regression também é instrutivo: mesma média e mesmo desvio não significam que os modelos aprenderam a mesma fronteira de decisão. Apenas significa que, neste experimento pequeno, o desempenho agregado coincidiu.

Os tempos também devem ser lidos em contexto. Todos são muito pequenos em termos absolutos, então aqui não faz sentido escolher um modelo apenas por diferenças de milissegundos.


## 6. O que muda entre os três modelos?

### Multinomial Naive Bayes
Muito rápido e historicamente forte para contagens e TF-IDF em texto.

### Logistic Regression
Modelo linear discriminativo que aprende pesos por feature e pode fornecer probabilidades via `predict_proba()`.

### LinearSVC
Procura uma fronteira linear com margem ampla. Costuma ser um baseline muito competitivo em espaços textuais de alta dimensionalidade.


## 7. Treinando o melhor candidato no dataset completo

Para fins didáticos, vamos selecionar o modelo com maior média de F1 macro na validação cruzada e treiná-lo em todos os exemplos.

Isso **não** substitui um conjunto de teste independente em um projeto real.


In [ ]:
best_name = results.iloc[0]["model"]
best_pipeline = pipelines[best_name]

best_pipeline.fit(texts, labels)

print("Melhor baseline na CV:", best_name)


## 8. Testando mensagens novas


In [ ]:
new_texts = [
    "como faço para alterar minha senha",
    "o suporte foi excelente",
    "estou muito irritado com o atraso",
]

predictions = best_pipeline.predict(new_texts)

for text, prediction in zip(new_texts, predictions):
    print(f"{prediction:12} | {text}")


## 9. Interpretando features em modelos lineares

Logistic Regression e LinearSVC aprendem pesos associados às features. Isso permite inspecionar quais termos/n-grams empurram a decisão em direção a cada classe.

Vamos usar `LinearSVC` para essa inspeção, independentemente de ele ter sido o vencedor da CV.


> 📘 **Glossário em contexto:** nesta seção, `coeficiente` e `margem` deixam de ser apenas definições e passam a aparecer como propriedades observáveis de modelos lineares.


In [ ]:
import numpy as np

interpret_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LinearSVC(random_state=SEED)),
])

interpret_pipeline.fit(texts, labels)

vectorizer = interpret_pipeline.named_steps["tfidf"]
classifier = interpret_pipeline.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

for class_index, class_name in enumerate(classifier.classes_):
    top_indices = np.argsort(classifier.coef_[class_index])[-8:][::-1]
    top_features = feature_names[top_indices]
    print(f"\nClasse: {class_name}")
    print(", ".join(top_features))


### Cuidado com a interpretação

Um peso alto não significa causalidade nem 'explicação completa' do modelo.

Ele mostra apenas que aquela feature contribui fortemente para a fronteira linear aprendida naquele corpus.


### O que as features mais fortes estão nos dizendo?

A inspeção do `LinearSVC` produziu padrões bastante intuitivos:

```text
duvida
→ como, onde, posso, fatura, cancelo serviço

elogio
→ excelente, satisfeito, rapidamente, equipe, rápido eficiente

reclamacao
→ não, péssimo, muito irritado, insatisfeito
```

Isso ajuda a auditar o comportamento do modelo porque conseguimos enxergar quais features empurram a decisão em direção a cada classe.

Mas cuidado com a interpretação:

- `não` aparecer em reclamação não significa que toda frase com `não` seja reclamação;
- `excelente` ter peso alto não significa que o modelo entenda elogio como uma pessoa;
- os pesos refletem padrões do corpus e da representação TF-IDF;
- features correlacionadas podem compartilhar informação;
- um coeficiente alto não prova causalidade.

Esse tipo de transparência lexical é uma vantagem prática dos modelos lineares: eles permitem uma auditoria simples que é muito mais difícil em Transformers.


## 10. Clássico versus Transformer

Agora podemos colocar a Aula 11 em perspectiva.

| Critério | TF-IDF + modelo linear | Transformer fine-tuned |
| --- | --- | --- |
| Treino | muito rápido | mais caro |
| CPU | geralmente suficiente | possível, mas mais lento |
| Memória | baixa/moderada | alta |
| Interpretabilidade lexical | maior | menor |
| Contexto semântico | limitado | muito maior |
| Dependência de modelo externo | não | sim |
| Baseline de produção | excelente | depende do ganho obtido |

A escolha deve ser orientada por evidência, não por prestígio tecnológico.


### O que realmente muda na decisão de arquitetura?

A comparação entre modelo linear e Transformer não deve ser lida como uma disputa de 'modelo antigo versus modelo moderno'.

Ela representa um trade-off de engenharia:

```text
TF-IDF + modelo linear
→ barato
→ rápido
→ fácil de inspecionar
→ contexto limitado

Transformer fine-tuned
→ maior capacidade contextual
→ maior custo
→ mais memória
→ menor transparência lexical
```

Por isso, a pergunta correta em produção é:

> **o ganho adicional do Transformer é grande o suficiente para justificar custo, latência, complexidade e manutenção?**

Se o baseline clássico já entrega desempenho suficiente para o objetivo de negócio, ele pode ser a melhor escolha.

Se o problema exige compreensão contextual que o TF-IDF não consegue capturar, então o Transformer pode justificar a complexidade adicional.

A escolha madura é orientada por evidência, não por novidade tecnológica.


## 11. Exercício guiado

Crie um pipeline com:

- `TfidfVectorizer(ngram_range=(1, 2))`;
- `LogisticRegression(max_iter=1000, random_state=SEED)`.

Depois:

1. avalie com a mesma validação cruzada `cv`;
2. use `f1_macro`;
3. imprima a média e o desvio-padrão;
4. compare com a linha de `LogisticRegression` na tabela `results`.


In [ ]:
# Escreva sua solução aqui.

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q12.hint()` e `q12.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q12 = TILExercise(
    hint_text=(
        "Use `Pipeline`, `TfidfVectorizer`, `LogisticRegression` e `cross_validate`. "
        "O resultado está em `scores['test_score']`; calcule `.mean()` e `.std()`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "exercise_pipeline = Pipeline([\n"
        "    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),\n"
        "    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED)),\n"
        "])\n\n"
        "scores = cross_validate(\n"
        "    exercise_pipeline, texts, labels, cv=cv, scoring='f1_macro'\n"
        ")\n\n"
        "print('Média:', round(scores['test_score'].mean(), 4))\n"
        "print('Desvio-padrão:', round(scores['test_score'].std(), 4))\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q12.hint() ou q12.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q12.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q12.solution()


## 12. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `scikit-learn`, `pandas`, `numpy`;
- representação: TF-IDF com unigramas + bigramas;
- modelos: MultinomialNB, LogisticRegression, LinearSVC;
- validação: StratifiedKFold com 4 folds;
- seed: 42;
- métrica: F1 macro;
- internet: desabilitada;
- dataset externo: nenhum.


## 13. Resumo

Nesta aula, você aprendeu que:

- baselines fortes são essenciais mesmo na era dos Transformers;
- Naive Bayes, Logistic Regression e LinearSVC podem ser muito competitivos em texto;
- uma comparação justa mantém representação, métrica e folds constantes;
- desempenho deve ser analisado junto com variabilidade e custo;
- modelos lineares permitem alguma inspeção lexical das features;
- complexidade adicional só se justifica quando produz ganho relevante.

### Ideia principal

```text
Antes de perguntar 'qual é o modelo mais moderno?',
pergunte 'qual é o modelo mais simples que resolve bem o problema?'
```

**Fim da Aula 12.**
